In [ ]:
!pip install gym

**Question 1: Train a CartPole Agent
Dataset Problem: Use the OpenAI Gym's CartPole-v1 environment to train an agent using a simple reinforcement learning algorithm.  Assume hyperparameters, as per requirement and develop the model. Try to apply the concepts discussed in class**

In [ ]:
import gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from collections import deque
import random


# Environment and training parameters
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# Hyperparameters
gamma = 0.99               # Discount factor
epsilon = 1.0              # Exploration rate
epsilon_min = 0.01         # Minimum exploration rate
epsilon_decay = 0.995      # Exploration decay rate
learning_rate = 0.001      # Learning rate
batch_size = 64            # Mini-batch size
max_episodes = 1000        # Total episodes to train
max_steps = 500            # Max steps per episode
memory_size = 2000         # Replay memory size

# 4. Define the Q-Network

def build_model():
    model = Sequential([
        Dense(24, input_dim=state_size, activation='relu'),
        Dense(24, activation='relu'),
        Dense(action_size, activation='linear')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='mse')
    return model



/usr/local/lib/python3.10/dist-packages/tensorflow/lite/python/util.py:55: DeprecationWarning: jax.xla_computation is deprecated. Please use the AOT APIs; see https://jax.readthedocs.io/en/latest/aot.html. For example, replace xla_computation(f)(*xs) with jit(f).lower(*xs).compiler_ir('hlo'). See CHANGELOG.md for 0.4.30 for more examples.
  from jax import xla_computation as _xla_computation
/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [ ]:
# 5. Implement Replay Memory

class ReplayMemory:
    def __init__(self, max_size): # Changed _init_ to __init__
        self.memory = deque(maxlen=max_size)

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def size(self):
        return len(self.memory)

In [ ]:
# 6. Define Epsilon-Greedy Policy

def epsilon_greedy_policy(state, epsilon, model):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)  # Explore
    # Reshape state to (1, state_size) before prediction
    q_values = model.predict(state.reshape(1, state_size))
    return np.argmax(q_values[0])  # Exploit

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
# 7. Train the Agent

# Initialize
model = build_model()
target_model = build_model()
target_model.set_weights(model.get_weights())
memory = ReplayMemory(memory_size)

for episode in range(max_episodes):
    state = env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for step in range(max_steps):
        # Choose action
        action = epsilon_greedy_policy(state, epsilon, model)

        # Take action and observe
        next_state, reward, done, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_size])

        # Store in memory
        memory.add(state, action, reward, next_state, done)

        # Train the model
        if memory.size() >= batch_size:
            minibatch = memory.sample(batch_size)
            for s, a, r, s_next, d in minibatch:
                target = r
                if not d:
                    target += gamma * np.amax(target_model.predict(s_next)[0])
                q_values = model.predict(s)
                q_values[0][a] = target
                model.fit(s, q_values, epochs=1, verbose=0)

        state = next_state
        total_reward += reward

        if done:
            print(f"Episode: {episode}/{max_episodes}, Reward: {total_reward}, Epsilon: {epsilon:.2f}")
            break

    # Decay epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    # Update target network
    if episode % 10 == 0:
        target_model.set_weights(model.get_weights())

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode: 0/1000, Reward: 36.0, Epsilon: 1.00
Episode: 1/1000, Reward: 16.0, Epsilon: 0.99
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/ste

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [6]:
#8. Test the Agent

state = env.reset()
state = np.reshape(state, [1, state_size])
done = False
total_reward = 0

while not done:
    env.render()
    action = np.argmax(model.predict(state)[0])
    state, reward, done, _ = env.step(action)
    state = np.reshape(state, [1, state_size])
    total_reward += reward

print(f"Test Reward: {total_reward}")
env.close()

/usr/local/lib/python3.10/dist-packages/gym/core.py:49: DeprecationWarning: WARN: You are calling render method, but you didn't specified the argument render_mode at environment initialization. To maintain backward compatibility, the environment will render in human mode.
If you want to render in human mode, initialize the environment in this way: gym.make('EnvName', render_mode='human') and don't call the render method.
See here for more information: https://www.gymlibrary.ml/content/api/
  deprecation(
/usr/local/lib/python3.10/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_reso

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


/usr/local/lib/python3.10/dist-packages/gym/core.py:49: DeprecationWarning: WARN: You are calling render method, but you didn't specified the argument render_mode at environment initialization. To maintain backward compatibility, the environment will render in human mode.
If you want to render in human mode, initialize the environment in this way: gym.make('EnvName', render_mode='human') and don't call the render method.
See here for more information: https://www.gymlibrary.ml/content/api/
  deprecation(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━

In [9]:
#----------------------------------------------------------#

# Question 2: Mountain Car with Q-Learning Dataset Problem: Use OpenAI Gym's MountainCar-v0 environment to train a Q-learning agent.Similar to the CartPole example, but with the Mountain Car environment. The Q-learning code will be similar, with adjustments to the state and action space to fit the Mountain Car environment.



1. Understanding the Environment

MountainCar-v0: The agent must drive the car up a mountain by accelerating left, right, or staying idle.

State space: Continuous with two dimensions:

Position (x) ∈ [-1.2, 0.6]

Velocity (v) ∈ [-0.07, 0.07]


Action space: Discrete with 3 actions:

0: Push left

1: No push

2: Push right


Goal: Get the car to a position ≥ 0.5.

In [21]:
# 2. Discretizing the State Space
num_bins = 10
position_bins = np.linspace(-1.2, 0.6, num_bins)
velocity_bins = np.linspace(-0.07, 0.07, num_bins)


def discretize_state(state, position_bins, velocity_bins):
    pos, vel = state
    pos_bin = np.digitize(pos, position_bins) - 1
    vel_bin = np.digitize(vel, velocity_bins) - 1
    return (pos_bin, vel_bin)


import gym
import numpy as np

# Hyperparameters
alpha = 0.1          # Learning rate
gamma = 0.99         # Discount factor
epsilon = 1.0        # Exploration rate
epsilon_decay = 0.99
epsilon_min = 0.1
episodes = 5000      # Total episodes
num_bins = 20        # Number of bins for discretizing state space

# Environment
env = gym.make("MountainCar-v0")
position_bins = np.linspace(-1.2, 0.6, num_bins)
velocity_bins = np.linspace(-0.07, 0.07, num_bins)

# Q-table initialization
q_table = np.random.uniform(low=-1, high=0, size=(num_bins, num_bins, env.action_space.n))

# Discretize state function
def discretize_state(state):
    pos, vel = state
    pos_bin = np.digitize(pos, position_bins) - 1
    vel_bin = np.digitize(vel, velocity_bins) - 1
    return (pos_bin, vel_bin)

# Training the agent
# Training the agent

total_rewards = []
for episode in range(episodes):
    # Access the 'observation' key to get the state
    # The issue was env.reset() returns a dictionary with 'observation' key
    # Get the observation from the dictionary returned by env.reset()
    state = discretize_state(env.reset())
    done = False
    total_reward = 0

    while not done:
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        # Take action and access the 'observation' key for the next state
        # The issue was env.step() returns a dictionary with 'observation' key
        # Get the observation from the dictionary returned by env.step()
        # Changed this line to unpack 4 values instead of 5
        next_state_raw, reward, done, _ = env.step(action)
        next_state = discretize_state(next_state_raw)

        # Q-value update
        best_next_action = np.argmax(q_table[next_state])
        td_target = reward + gamma * q_table[next_state][best_next_action]
        q_table[state][action] += alpha * (td_target - q_table[state][action])

        state = next_state
        total_reward += reward
        total_rewards.append(total_reward)  # Add total_reward to total_rewards after each episode


    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Log progress
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}, Total Reward: {total_reward}, Epsilon: {epsilon:.2f}")

# Close environment
env.close()
# Calculate and print the average reward and accuracy (success rate)
avg_reward = np.mean(total_rewards)
num_test_episodes = len(total_rewards)  # Get the number of episodes from total_rewards length
success_rate = sum(1 for reward in total_rewards if reward >= -200) / num_test_episodes

print(f"\nAverage Reward over {num_test_episodes} episodes: {avg_reward}")
print(f"Success Rate: {success_rate:.2%}")

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode 100, Total Reward: -200.0, Epsilon: 0.37
Episode 200, Total Reward: -200.0, Epsilon: 0.13
Episode 300, Total Reward: -200.0, Epsilon: 0.10
Episode 400, Total Reward: -200.0, Epsilon: 0.10
Episode 500, Total Reward: -200.0, Epsilon: 0.10
Episode 600, Total Reward: -200.0, Epsilon: 0.10
Episode 700, Total Reward: -200.0, Epsilon: 0.10
Episode 800, Total Reward: -200.0, Epsilon: 0.10
Episode 900, Total Reward: -200.0, Epsilon: 0.10
Episode 1000, Total Reward: -200.0, Epsilon: 0.10
Episode 1100, Total Reward: -200.0, Epsilon: 0.10
Episode 1200, Total Reward: -200.0, Epsilon: 0.10
Episode 1300, Total Reward: -166.0, Epsilon: 0.10
Episode 1400, Total Reward: -200.0, Epsilon: 0.10
Episode 1500, Total Reward: -200.0, Epsilon: 0.10
Episode 1600, Total Reward: -200.0, Epsilon: 0.10
Episode 1700, Total Reward: -200.0, Epsilon: 0.10
Episode 1800, Total Reward: -200.0, Epsilon: 0.10
Episode 1900, Total Reward: -200.0, Epsilon: 0.10
Episode 2000, Total Reward: -160.0, Epsilon: 0.10
Episode 2